# Lab 4: Implementation of Dynamic Programming

## Objective of the Lab

- To understand the concept of **Dynamic Programming (DP)** as an algorithm design paradigm.
- To identify problems that exhibit **optimal substructure** and **overlapping subproblems**.
- To implement classical dynamic programming algorithms and analyze how they avoid redundant computation compared to naive recursive approaches.

## Related Theory

**Dynamic Programming** is an algorithm design technique for solving problems by breaking them into overlapping subproblems, solving each subproblem only once, and storing (memoizing) its result for reuse. A problem is suited to DP if it exhibits:

1. **Optimal Substructure** — an optimal solution to the problem can be constructed from optimal solutions of its subproblems.
2. **Overlapping Subproblems** — the same subproblems are solved repeatedly if approached with plain recursion; DP stores these results (in a table or memo) to avoid recomputation.

DP can be implemented **top-down** (recursion + memoization) or **bottom-up** (iterative table filling), and typically improves an exponential-time naive recursive solution to polynomial time.

- **Floyd-Warshall Algorithm (All Pairs Shortest Path)**: Computes the shortest distance between every pair of vertices in a weighted graph by progressively considering each vertex as an intermediate point. Runs in O(V³) time.
- **Travelling Salesman Problem (TSP)**: Finds the minimum-cost Hamiltonian cycle visiting every city exactly once. The DP (Held-Karp) approach uses bitmasking to represent visited-city subsets, reducing the naive O(n!) brute force to O(n² · 2ⁿ).
- **String Editing (Edit Distance)**: Computes the minimum number of insertions, deletions, and substitutions to transform one string into another, using a 2D DP table over string prefixes.
- **0/1 Knapsack Problem**: Given items with weights and values and a knapsack of fixed capacity, finds the maximum value achievable without exceeding capacity, where each item is either taken whole or not at all (unlike the fractional version, this requires DP, not greedy).
- **Matrix Chain Multiplication**: Determines the optimal order (parenthesization) to multiply a chain of matrices so as to minimize the total number of scalar multiplications, using interval DP.
- **Flow Shop Scheduling**: Given n jobs that must each pass through the same sequence of machines, determines a job order that minimizes the makespan (total completion time). The DP formulation here uses bitmasking over the set of scheduled jobs.

## Related Diagram

**DP table fill direction — Edit Distance (string "cat" -> "cut"):**

```
        ""   c   u   t
   ""    0   1   2   3
   c     1   0   1   2
   a     2   1   1   2
   t     3   2   2   1
```
Each cell dp[i][j] depends only on dp[i-1][j], dp[i][j-1], and dp[i-1][j-1] — filled left-to-right, top-to-bottom.

**Matrix Chain Multiplication — interval DP:**

```
dp[i][j] = min cost to multiply matrices i..j
dp[i][j] = min over k in [i, j-1] of:
           dp[i][k] + dp[k+1][j] + (cost of multiplying the two resulting matrices)

Filled diagonally by increasing chain length:
  length 1: dp[i][i] = 0
  length 2: dp[i][i+1]
  length 3: dp[i][i+2]
  ...
```

**TSP / Flow Shop bitmask DP state:**

```
dp[mask][i] = best value considering the subset of jobs/cities in 'mask',
              currently ending at job/city i.
mask is a bitmask, e.g. 0b0101 means jobs {0, 2} have been processed.
```

## Computer Code

### 1. All Pairs Shortest Path — Floyd-Warshall Algorithm

In [1]:
def floyd_warshall(graph):
    """
    graph: n x n adjacency matrix, graph[i][j] = weight of edge i->j,
           or float('inf') if no direct edge (0 on the diagonal).
    Returns the matrix of shortest distances between every pair of vertices.
    """
    n = len(graph)
    dist = [row[:] for row in graph]

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]

    return dist

INF = float('inf')
graph = [
    [0,   5,   INF, 10],
    [INF, 0,   3,   INF],
    [INF, INF, 0,   1],
    [INF, INF, INF, 0],
]

result = floyd_warshall(graph)
print("Shortest distance matrix (all pairs):")
for row in result:
    print(["INF" if x == INF else x for x in row])

Shortest distance matrix (all pairs):
[0, 5, 8, 9]
['INF', 0, 3, 4]
['INF', 'INF', 0, 1]
['INF', 'INF', 'INF', 0]


### 2. Travelling Salesman Problem (Held-Karp DP)

In [2]:
def tsp_dp(dist):
    """
    dist: n x n distance matrix.
    Returns the minimum cost of a tour visiting all cities starting and ending at city 0.
    """
    n = len(dist)
    all_visited = (1 << n) - 1
    memo = {}

    def visit(mask, pos):
        if mask == all_visited:
            return dist[pos][0]
        if (mask, pos) in memo:
            return memo[(mask, pos)]

        best = float('inf')
        for city in range(n):
            if not (mask & (1 << city)):
                new_cost = dist[pos][city] + visit(mask | (1 << city), city)
                best = min(best, new_cost)

        memo[(mask, pos)] = best
        return best

    return visit(1, 0)

dist = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0],
]

min_cost = tsp_dp(dist)
print(f"Distance matrix: {dist}")
print(f"Minimum cost of the TSP tour: {min_cost}")

Distance matrix: [[0, 10, 15, 20], [10, 0, 35, 25], [15, 35, 0, 30], [20, 25, 30, 0]]
Minimum cost of the TSP tour: 80


### 3. String Editing — Edit Distance

In [3]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],     # deletion
                    dp[i][j - 1],     # insertion
                    dp[i - 1][j - 1]  # substitution
                )

    return dp[m][n]

s1, s2 = "cat", "cut"
result = edit_distance(s1, s2)
print(f"String 1: '{s1}'")
print(f"String 2: '{s2}'")
print(f"Minimum edit distance: {result}")

String 1: 'cat'
String 2: 'cut'
Minimum edit distance: 1


### 4. 0/1 Knapsack Problem (Dynamic Programming)

In [4]:
def knapsack_01(weights, values, capacity):
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            if weights[i - 1] <= w:
                dp[i][w] = max(
                    dp[i - 1][w],
                    values[i - 1] + dp[i - 1][w - weights[i - 1]]
                )
            else:
                dp[i][w] = dp[i - 1][w]

    return dp[n][capacity]

weights = [1, 3, 4, 5]
values = [1, 4, 5, 7]
capacity = 7

max_value = knapsack_01(weights, values, capacity)
print(f"Weights: {weights}")
print(f"Values: {values}")
print(f"Knapsack capacity: {capacity}")
print(f"Maximum value obtainable: {max_value}")

Weights: [1, 3, 4, 5]
Values: [1, 4, 5, 7]
Knapsack capacity: 7
Maximum value obtainable: 9


### 5. Matrix Chain Multiplication

In [5]:
def matrix_chain_order(dims):
    """
    dims: list where matrix i has dimensions dims[i-1] x dims[i]
    Returns the minimum number of scalar multiplications needed.
    """
    n = len(dims) - 1  # number of matrices
    dp = [[0] * n for _ in range(n)]

    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i][j] = float('inf')
            for k in range(i, j):
                cost = (dp[i][k] + dp[k + 1][j] +
                        dims[i] * dims[k + 1] * dims[j + 1])
                if cost < dp[i][j]:
                    dp[i][j] = cost

    return dp[0][n - 1]

# 4 matrices: A1(10x30), A2(30x5), A3(5x60)... dims represents chained dimensions
dims = [10, 30, 5, 60]
min_ops = matrix_chain_order(dims)
print(f"Matrix dimensions chain: {dims}")
print(f"Minimum number of scalar multiplications: {min_ops}")

Matrix dimensions chain: [10, 30, 5, 60]
Minimum number of scalar multiplications: 4500


### 6. Flow Shop Scheduling

Given n jobs that must each be processed on the same sequence of machines (in the same order), this bitmask DP finds the job sequence that minimizes the total makespan (completion time of the last job on the last machine).

In [6]:
def flow_shop_scheduling(processing_times):
    """
    processing_times: list of lists, processing_times[j] = [time on machine 1, machine 2, ...]
    Returns the minimum makespan using bitmask DP over job permutations.
    """
    n = len(processing_times)
    num_machines = len(processing_times[0])
    full_mask = (1 << n) - 1

    # dp[mask] = list of completion times on each machine for the best sequence
    # covering the jobs in 'mask'
    memo = {}

    def solve(mask):
        if mask in memo:
            return memo[mask]
        if mask == 0:
            return [0] * num_machines

        best_completion = None
        for job in range(n):
            bit = 1 << job
            if mask & bit:
                prev_completion = solve(mask ^ bit)
                completion = [0] * num_machines
                completion[0] = prev_completion[0] + processing_times[job][0]
                for m in range(1, num_machines):
                    completion[m] = (max(completion[m - 1], prev_completion[m])
                                      + processing_times[job][m])
                if best_completion is None or completion[-1] < best_completion[-1]:
                    best_completion = completion

        memo[mask] = best_completion
        return best_completion

    result = solve(full_mask)
    return result[-1]  # makespan = completion time on the last machine

# 4 jobs, each with processing times on 2 machines
processing_times = [
    [5, 2],
    [1, 6],
    [4, 3],
    [3, 4],
]

makespan = flow_shop_scheduling(processing_times)
print(f"Job processing times (machine1, machine2): {processing_times}")
print(f"Minimum makespan: {makespan}")

Job processing times (machine1, machine2): [[5, 2], [1, 6], [4, 3], [3, 4]]
Minimum makespan: 16


## Analysis of the Algorithms

- **Floyd-Warshall**: O(V³) time and O(V²) space. Simple triple-nested loop that computes all-pairs shortest paths in one pass, even with negative edge weights (as long as there are no negative cycles) — more efficient than running Dijkstra's algorithm from every vertex when the graph is dense.
- **TSP (Held-Karp DP)**: O(n² · 2ⁿ) time and O(n · 2ⁿ) space, a major improvement over the O(n!) brute-force approach, though still exponential — practical only for a moderate number of cities (roughly n ≤ 20).
- **Edit Distance**: O(m·n) time and space for strings of length m and n, compared to exponential time for the naive recursive solution without memoization, since many overlapping subproblems (common prefixes) are shared.
- **0/1 Knapsack**: O(n · W) time and space, where W is the capacity — pseudo-polynomial, since it depends on the numeric value of W, not just the number of bits needed to represent it.
- **Matrix Chain Multiplication**: O(n³) time and O(n²) space, filling the DP table by increasing chain length; avoids the exponential number of possible parenthesizations (Catalan number growth) of the naive approach.
- **Flow Shop Scheduling**: O(n² · 2ⁿ) time and O(n · 2ⁿ) space using bitmask DP, again a major improvement over the O(n!) brute-force search over all job permutations, but still exponential in the number of jobs.

## Discussion and Conclusion

This lab demonstrated how Dynamic Programming systematically improves on naive recursive or brute-force solutions by identifying and reusing overlapping subproblems. Floyd-Warshall, Edit Distance, 0/1 Knapsack, and Matrix Chain Multiplication all achieved polynomial (or pseudo-polynomial) time complexity by filling a DP table bottom-up, where each cell is computed once from previously solved subproblems.

The Travelling Salesman Problem and Flow Shop Scheduling illustrated a different aspect of DP: even when a problem is fundamentally NP-Hard, dynamic programming (via bitmasking over subsets) can still meaningfully reduce the complexity from factorial time, O(n!), to exponential time, O(n² · 2ⁿ) — a substantial improvement, though still impractical for very large inputs.

A recurring theme across all six problems was the trade-off between time and space: DP typically trades additional memory (for the table or memo) in exchange for avoiding redundant recomputation, which is worthwhile whenever a problem exhibits optimal substructure and overlapping subproblems.

Overall, this lab reinforced that dynamic programming is one of the most broadly applicable algorithm design techniques, spanning graph algorithms (Floyd-Warshall, TSP), string algorithms (Edit Distance), combinatorial optimization (Knapsack, Matrix Chain), and scheduling (Flow Shop) — all unified by the same core idea of solving each subproblem exactly once.